In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.8 MB/s eta 0:00:00


In [3]:
import faiss

print(faiss.__version__)

1.15.0


In [4]:
import faiss
import numpy as np
import pandas as pd
from pathlib import Path

In [5]:
PROJECT_PATH = Path("/content/drive/MyDrive/uterine-emg-rag")

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"

In [6]:
embeddings = np.load(
    EMBEDDINGS_PATH / "embeddings.npy"
)

In [7]:
print(embeddings.shape)

(833, 384)


In [8]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

In [9]:
metadata.head()

,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966


In [10]:
assert len(embeddings) == len(metadata)

print("Embeddings:", len(embeddings))
print("Metadata:", len(metadata))

Embeddings: 833
Metadata: 833


In [11]:
embeddings = embeddings.astype("float32")

In [12]:
print(embeddings.dtype)

float32


In [13]:
faiss.normalize_L2(embeddings)

In [14]:
dimension = embeddings.shape[1]

print(dimension)

384


In [15]:
index = faiss.IndexFlatIP(dimension)

In [16]:
index.add(embeddings)

In [17]:
print(index.ntotal)

833


In [18]:
FAISS_PATH = PROJECT_PATH / "faiss_index"

FAISS_PATH.mkdir(exist_ok=True)

In [19]:
faiss.write_index(
    index,
    str(FAISS_PATH / "uterine_emg.index")
)

In [20]:
loaded_index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

In [21]:
print(loaded_index.ntotal)

833


In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
query = "What changes occur in uterine electrical activity before labor?"

In [24]:
query_embedding = model.encode(
    [query]
)

In [25]:
query_embedding = query_embedding.astype("float32")

In [26]:
faiss.normalize_L2(query_embedding)

In [27]:
k = 5

scores, indices = loaded_index.search(
    query_embedding,
    k
)

In [28]:
print(scores)
print(indices)

[[0.7379843  0.7183577  0.70589936 0.69237417 0.6916882 ]]
[[640  39 550 230 273]]


In [29]:
for rank, idx in enumerate(indices[0], start=1):

    print(f"\n--- Result {rank} ---")

    print("Score:", scores[0][rank - 1])

    print("Paper:", metadata.iloc[idx]["paper"])

    print("Chunk ID:", metadata.iloc[idx]["chunk_id"])

    print("Text:")
    print(metadata.iloc[idx]["text"])


--- Result 1 ---
Score: 0.7379843
Paper: paper7
Chunk ID: 33
Text:
only measure the onset of labor indirectly and do not detect cellular changes characteristic of true labor, so that 
their predictive values for term or preterm delivery are poor. Both term and preterm births involve activation of 
the myometrium. Several events in the uterine muscle precede labor: cell excitability increases due to changes 
in transduction mechanisms and synthesis of various proteins, including ion channels and receptors for 
uterotonins (Fuchs et al 1984, Tezuka et al 1995). At the same time, the factors that inhibit myometrial activity, 
such as the nitric oxide system, are downregulated, leading to withdrawal of uterine relaxation (Garfield et al 
1998). Electrical coupling between myometrial cells also increases, and an electrical syncytium allowing the 
propagation of action potentials from cell to cell is formed (Leitich et al 1999, Honest et al 2002). These changes 
are required for effective c

In [30]:
def semantic_search(query, model, index, metadata, k=5):

    # 1. Convert query to embedding
    query_embedding = model.encode(
        [query]
    ).astype("float32")

    # 2. Normalize
    faiss.normalize_L2(query_embedding)

    # 3. Search
    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    # 4. Collect results
    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [31]:
results = semantic_search(
    "What changes occur in uterine electrical activity before labor?",
    model,
    loaded_index,
    metadata,
    k=5
)

In [32]:
for result in results:

    print(f"\nRank: {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Paper: {result['paper']}")
    print(f"Chunk: {result['chunk_id']}")
    print(result["text"])


Rank: 1
Score: 0.7380
Paper: paper7
Chunk: 33
only measure the onset of labor indirectly and do not detect cellular changes characteristic of true labor, so that 
their predictive values for term or preterm delivery are poor. Both term and preterm births involve activation of 
the myometrium. Several events in the uterine muscle precede labor: cell excitability increases due to changes 
in transduction mechanisms and synthesis of various proteins, including ion channels and receptors for 
uterotonins (Fuchs et al 1984, Tezuka et al 1995). At the same time, the factors that inhibit myometrial activity, 
such as the nitric oxide system, are downregulated, leading to withdrawal of uterine relaxation (Garfield et al 
1998). Electrical coupling between myometrial cells also increases, and an electrical syncytium allowing the 
propagation of action potentials from cell to cell is formed (Leitich et al 1999, Honest et al 2002). These changes 
are required for effective contractions that end 

In [33]:
queries = [
    "What are the characteristics of uterine electromyography signals?",

    "How is uterine electrical activity related to labor?",

    "What signal processing methods are used for uterine EMG?",

    "What features are extracted from uterine EMG signals?",

    "How can uterine EMG predict the onset of labor?",

    "What is the relationship between uterine contractions and EMG activity?"
]

In [34]:
for query in queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = semantic_search(
        query,
        model,
        loaded_index,
        metadata,
        k=3
    )

    for result in results:

        print(
            f"\n[{result['rank']}] "
            f"Score={result['score']:.4f}"
        )

        print(result["text"][:500])


QUERY: What are the characteristics of uterine electromyography signals?

[1] Score=0.7297
unfamiliar with EHG tracings, integration into monitors and records is incomplete, and,
as with any diagnostic test, false positives or negatives carry clinical and ethical costs;
large-scale validation and training are prerequisites for routine use.
3. Electromyography (EMG) for Uterine Contractility
EMG, in the context of uterine contractility, encompasses techniques that measure
the electrical activity generated by uterine smooth muscle. While EHG represents the
noninvasive, surface-based for

[2] Score=0.7074
Hertsch first recorded electrical signals from the human uterus during labor in 1950, coining
the term “electrohysterograph” for this technique [6]. Subsequent studies from the period
1950–1970 confirmed that bursts of bioelectrical activity accompany uterine contractions.
For example, Larks and Dasgupta (1958) characterized EHG waveforms during pregnancy
and labor [7], and Wolfs and va